# What's in an Avocado Toast: A Supply Chain Analysis

![](avocado_wallpaper.jpeg)

You find yourself in London, crafting a delectable avocado toast, a dish that has risen dramatically in popularity on breakfast menus since the 2010s. This straightforward recipe requires just five ingredients: a ripe avocado, half a lemon, a generous pinch of salt flakes, two slices of sourdough bread, and a good drizzle of extra virgin olive oil. Most of these ingredients are now staples in grocery stores, and as you will find with this project, that is no small feat!

In this project, you'll conduct a supply chain analysis of three ingredients used in avocado toast using the Open Food Facts database. This database contains extensive, openly-sourced information on various foods, including their origins. Through this analysis, you will gain an in-depth understanding of the complex supply chain involved in producing a single dish.

Three pairs of files are provided in the data folder:
- A CSV file for each ingredient, such as `avocado.csv`, with data about each food item and countries of origin.
- A TXT file for each ingredient, such as `relevant_avocado_categories`, containing only the category tags of interest for that food.

Here are some other key points about these files:
- Some of the rows of data in each of the three CSV files do not contain relevant data for your investigation. In each dataset, you will need to filter out rows with irrelevant data, based on values in the `categories_tags` column. Examples of categories are fruits, vegetables, and fruit-based oils. Filter the DataFrame to include only rows where `categories_tags` contains one of the tags in the relevant categories for that ingredient.
- Each row of data usually has multiple category tags in the `categories_tags` column.
There is a column in each CSV file called `origins_tags`, which contains strings for the country of origin of each item.

After completing this project, you'll be armed with a list of ingredients and their countries of origin and be well-positioned to launch into other analyses that explore how long, on average, these ingredients spend at sea.

[Open Food Facts database](https://world.openfoodfacts.org/)

In [1]:
import pandas as pd

#################################################
# Read the avocado data

# Read file
avocado = pd.read_csv('data/avocado.csv',sep = '\t')

# Subset relevant columns
relevant_columns = ['code', 'lc', 'product_name_en', 'quantity', 'serving_size', 'packaging_tags', 'brands', 'brands_tags', 'categories_tags', 'labels_tags', 'countries', 'countries_tags', 'origins','origins_tags']
avocado = avocado[relevant_columns]

# Reading data from TXT files for relevant category tags
with open('data/relevant_avocado_categories.txt', "r") as file:
    relevant_avocado_categories = file.read().splitlines()
    file.close()

#################################################
# Filter avocado data using relevant category tags
# Turning a column of comma separated tags into a column of lists
avocado['categories_tags'] = avocado['categories_tags'].str.split(',')

# Dropping rows with null values in a particular column
avocado = avocado.dropna(subset = 'categories_tags')

# Filtering a DataFrame based on a column of lists
avocado = avocado[avocado['categories_tags'].apply(lambda x: any([i for i in x if i in relevant_avocado_categories]))]

#################################################
# Where do most UK avocados come from ?
# Filtering your DataFrame by a particular country
UK_avocados = avocado[(avocado['countries']=='United Kingdom')]

# Counting and ordering by the unique values in the country of origin column
COO_counts = UK_avocados['origins_tags'].value_counts()

# Get the country with the highest count
top_avocado_origin = (COO_counts.index[0])

# Strip out characters before country name
top_avocado_origin = top_avocado_origin.lstrip("en:")

# Replace hyphen in country name with a space, if needed
top_avocado_origin = top_avocado_origin.replace('-', ' ')

#################################################
# Create a user-defined function to call for each ingredient
# Creating function to call for each ingredient, performing same tasks in function as you did for the avocado process

def read_and_filter_data(filename, relevant_categories):
    df = pd.read_csv(filename,sep = '\t')
    relevant_columns = ['code', 'lc', 'product_name_en', 'quantity', 'serving_size', 'packaging_tags', 'brands', 'brands_tags', 'categories_tags', 'labels_tags', 'countries', 'countries_tags', 'origins','origins_tags']
    df = df[relevant_columns]

    with open(relevant_categories, "r") as file:
        relevant_categories = file.read().splitlines()
        file.close()
    
    df['categories_tags'] = df['categories_tags'].str.split(',')
    df = df.dropna(subset = 'categories_tags')
    df = df[df['categories_tags'].apply(lambda x: any([i for i in x if i in relevant_categories]))]

    UK_df = df[(df['countries']=='United Kingdom')]
    COO_counts = UK_df['origins_tags'].value_counts()
    country_of_origin = (COO_counts.index[0])
    country_of_origin = country_of_origin.lstrip("en:")
    country_of_origin = country_of_origin.replace('-', ' ')
    return country_of_origin

#################################################
# Read relevant categories data and call function for each ingredient

top_olive_oil_origin = read_and_filter_data('data/olive_oil.csv','data/relevant_olive_oil_categories.txt')
top_sourdough_origin = read_and_filter_data('data/sourdough.csv','data/relevant_sourdough_categories.txt')